In [ ]:
!pip install azure-storage-blob pandas numpy scikit-learn tensorflow

import os
import json
import pandas as pd
import numpy as np
from io import StringIO
from azure.storage.blob import BlobServiceClient
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization, Input
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from tqdm.notebook import tqdm
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from typing import Dict, Any, List
AZURE_CONNECTION_STRING = ""


In [ ]:
"""
Create a feature importance summary CSV using a rank-ensemble 'importance_score'.

Produces: feature_importance_summary.csv

Metrics combined:
 - variance (on standardized features)
 - ANOVA F-value (f_classif)
 - Mutual Information (mutual_info_classif)
 - RandomForest feature_importance (balanced RF on 70% train split)
 - max absolute per-class effect size (Cohen's d)

Behavior:
 - Auto-detects a target column by name heuristics (safety, bin, label, target, class).
 - If numeric with >20 unique values, it will bin into 5 quantiles.
 - Converts non-numeric feature columns to numeric (coerce), fills NaN with 0.0.
 - Adds simple category tags (obj / seg / other) based on feature name prefixes.
"""

import os
import re
import numpy as np
import pandas as pd
import warnings

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.feature_selection import f_classif, mutual_info_classif
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

# ----------------- CONFIG -----------------
FEATURES_CSV = "processed_features_conf_0.7.csv"   # input (features + possible target)
OUT_CSV = "feature_importance_summary.csv"         # output summary
RANDOM_STATE = 42
RF_ESTIMATORS = 200
RF_TEST_SIZE = 0.30   # train RF on 70% to reduce overfit
# ------------------------------------------

# helper: detect a target column
def detect_target_col(df):
    candidate_target_cols = [c for c in df.columns if any(k in c.lower() for k in ("safety", "label", "target", "bin", "class", "y"))]
    if not candidate_target_cols:
        return None
    # prefer specific names
    for prefer in ("safety_bin", "safety", "safety_score", "label", "target", "class", "bin"):
        for c in candidate_target_cols:
            if prefer in c.lower():
                return c
    return candidate_target_cols[0]

# Load
if not os.path.exists(FEATURES_CSV):
    raise FileNotFoundError(f"Features CSV not found: {FEATURES_CSV}")

df_full = pd.read_csv(FEATURES_CSV, index_col=0)
print("Loaded:", FEATURES_CSV, "shape:", df_full.shape)

# Detect target
target_col = detect_target_col(df_full)
if target_col is None:
    raise ValueError("No target column found by heuristics. Rename your target to include 'safety', 'bin', 'label', 'target' or 'class'.")

print("Using target column:", target_col)

# Split features and target
y_series = df_full[target_col].copy()
X_df = df_full.drop(columns=[target_col]).copy()

# Ensure features numeric: coerce non-numeric -> NaN -> fill 0
for c in X_df.columns:
    X_df[c] = pd.to_numeric(X_df[c], errors='coerce')
X_df = X_df.fillna(0.0)

# Prepare y (encode if necessary; bin if too many unique numeric values)
if pd.api.types.is_integer_dtype(y_series) or pd.api.types.is_float_dtype(y_series):
    nunique = int(y_series.nunique())
    print("Target numeric, unique values:", nunique)
    if nunique > 20:
        # bin into 5 quantiles
        y_binned, bins = pd.qcut(y_series, q=5, labels=False, retbins=True, duplicates='drop')
        y = y_binned.values
        class_names = [f"Bin_{i}" for i in range(len(np.unique(y)))]
        print("Binned numeric target into", len(class_names), "quantile classes.")
    else:
        le = LabelEncoder()
        y = le.fit_transform(y_series.values)
        class_names = [str(cl) for cl in le.classes_]
else:
    le = LabelEncoder()
    y = le.fit_transform(y_series.astype(str).values)
    class_names = [str(cl) for cl in le.classes_]

n_classes = len(np.unique(y))
print("Number of classes used:", n_classes)

# Standardize features for metrics that require it
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_df.values)  # ndarray shape (n_samples, n_features)
feat_names = X_df.columns.tolist()
n_feats = len(feat_names)
print("Number of features:", n_feats)

# Prepare results DataFrame
results = pd.DataFrame(index=feat_names)
results["variance"] = np.var(X_scaled, axis=0)
results["mean_abs"] = np.mean(np.abs(X_scaled), axis=0)

# Suppress known sklearn warnings for constant features during ANOVA
warnings.filterwarnings("ignore", message="Features .* are constant.")

# 1) ANOVA F-value
try:
    f_vals, p_vals = f_classif(X_scaled, y)
except Exception as e:
    # if something fails, populate NaNs
    f_vals = np.full(n_feats, np.nan)
    p_vals = np.full(n_feats, np.nan)
    print("f_classif failed:", e)
results["f_value"] = f_vals
results["f_pvalue"] = p_vals

# 2) Mutual Information
try:
    mi = mutual_info_classif(X_scaled, y, discrete_features=False, random_state=RANDOM_STATE)
except Exception as e:
    mi = np.full(n_feats, np.nan)
    print("mutual_info_classif failed:", e)
results["mutual_info"] = mi

# 3) Random Forest importance (train/test split to reduce overfit)
X_tr, X_te, y_tr, y_te = train_test_split(X_scaled, y, test_size=RF_TEST_SIZE,
                                          random_state=RANDOM_STATE, stratify=y)
rf = RandomForestClassifier(n_estimators=RF_ESTIMATORS, random_state=RANDOM_STATE,
                            class_weight='balanced', n_jobs=-1)
rf.fit(X_tr, y_tr)
rf_imp = rf.feature_importances_
results["rf_importance"] = rf_imp

# 4) Per-class effect sizes (Cohen's d) and aggregates
for cls in np.unique(y):
    cls_mask = (y == cls)
    others_mask = ~cls_mask
    # mean difference
    mean_cls = X_scaled[cls_mask].mean(axis=0)
    mean_oth = X_scaled[others_mask].mean(axis=0)
    # pooled std
    var_cls = X_scaled[cls_mask].var(axis=0, ddof=0)
    var_oth = X_scaled[others_mask].var(axis=0, ddof=0)
    pooled_std = np.sqrt((var_cls + var_oth) / 2.0)
    pooled_std[pooled_std == 0] = 1e-9
    cohens_d = (mean_cls - mean_oth) / pooled_std
    results[f"effect_size_class_{cls}"] = cohens_d

# Aggregate effect sizes
effect_cols = [c for c in results.columns if c.startswith("effect_size_class_")]
if effect_cols:
    abs_stack = np.abs(results[effect_cols].values)
    results["max_abs_effect_size"] = np.max(abs_stack, axis=1)
    results["mean_abs_effect_size"] = np.mean(abs_stack, axis=1)
else:
    results["max_abs_effect_size"] = 0.0
    results["mean_abs_effect_size"] = 0.0

# 5) Rank-ensemble -> importance_score
rank_cols = ["variance", "f_value", "mutual_info", "rf_importance", "max_abs_effect_size"]
# If some rank cols are missing (NaN), ranking will still work; fill NaN with lowest possible value before ranking
rtemp = results[rank_cols].copy()
# Replace -inf/inf with NaN, then fill NaN with very small numbers so they rank low
rtemp = rtemp.replace([np.inf, -np.inf], np.nan).fillna(0.0)

ranks = rtemp.rank(ascending=False, method='average', na_option='bottom')
results["mean_rank"] = ranks.mean(axis=1)

# Normalize mean_rank to 0..1, invert so higher importance -> closer to 1
min_r = results["mean_rank"].min()
max_r = results["mean_rank"].max()
if max_r == min_r:
    results["importance_score"] = 1.0
else:
    results["importance_score"] = 1.0 - ((results["mean_rank"] - min_r) / (max_r - min_r))

# Add simple category tags based on prefix
def categorize_feature(name):
    if re.match(r'^(obj_|object_|obj\.)', name, flags=re.I):
        return "object"
    if re.match(r'^(seg_|segmentation_|mask_|seg\.)', name, flags=re.I):
        return "segmentation"
    return "other"

results["category"] = [categorize_feature(n) for n in results.index]

# Reorder columns for readability
ordered_cols = ["category", "variance", "f_value", "f_pvalue", "mutual_info",
                "rf_importance", "max_abs_effect_size", "mean_abs_effect_size",
                "mean_rank", "importance_score"] + effect_cols
ordered_cols = [c for c in ordered_cols if c in results.columns]  # guard

results = results[ordered_cols]

# Save CSV
results.to_csv(OUT_CSV, index=True)
print("Saved importance summary to:", OUT_CSV)

# Print top 20 features
print("\nTop 20 by importance_score:")
print(results.sort_values("importance_score", ascending=False).head(20)[["importance_score", "category"]])


In [ ]:
# Paste this into your script. It will plot ALL features with their NAMES
# (y-axis labels), color-code by category (object / segmentation / other),
# annotate scores on bars for the top features, and save the figure.
#
# It is robust to different CSV formats:
# - If your CSV has a "feature" column it uses that as the name.
# - If not, it will try to use the index as feature names.
# - If there is no "category" column it will infer categories from name prefixes.

import os
import re
import math
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

IMPORTANCE_CSV = "feature_importance_summary.csv"
OUT_PLOT = "feature_importance_all_features_with_names.png"
TOP_ANNOTATE = 50   # annotate numeric score on top N bars (set 0 to disable)
MAX_LABELS_FOR_FULL_PLOT = 300  # if more features than this, show smaller font + optionally warn

if not os.path.exists(IMPORTANCE_CSV):
    raise FileNotFoundError(f"Importance CSV not found: {IMPORTANCE_CSV}")

# Load importance table (handle 'feature' col or index)
imp = pd.read_csv(IMPORTANCE_CSV)
if 'feature' in imp.columns:
    imp = imp.set_index('feature')
elif imp.columns[0] == 'Unnamed: 0' and imp['Unnamed: 0'].dtype == object:
    imp = imp.set_index('Unnamed: 0')

if 'importance_score' not in imp.columns:
    raise ValueError("importance_score column missing from importance CSV. Run the importance creation script first.")

# If category column present use it, otherwise infer from names
if 'category' in imp.columns:
    categories = imp['category'].astype(str).values
else:
    def infer_category(name):
        if re.match(r'^(obj_|object_|obj\.)', name, flags=re.I):
            return "object"
        if re.match(r'^(seg_|segmentation_|mask_|seg\.)', name, flags=re.I):
            return "segmentation"
        return "other"
    categories = [infer_category(n) for n in imp.index]

imp = imp.copy()
imp['category_inferred'] = categories

# Sort by importance
imp_sorted = imp.sort_values('importance_score', ascending=False)
feature_names = imp_sorted.index.tolist()
scores = imp_sorted['importance_score'].values
cats = imp_sorted['category_inferred'].values

# Color mapping (customize as you like)
color_map = {
    "object": "#1f77b4",       # blue
    "segmentation": "#2ca02c", # green
    "other": "#7f7f7f"         # gray
}
bar_colors = [color_map.get(c, "#7f7f7f") for c in cats]

# Plot sizing
n_feats = len(feature_names)
# Height per feature (in inches) — small enough to fit many features but readable
height_per_feature = 0.16
fig_height = max(6, min(140, height_per_feature * n_feats))  # cap so it doesn't explode
fig_width = 14

plt.figure(figsize=(fig_width, fig_height))
y_pos = np.arange(n_feats)

# Plot bars reversed so highest is on top
bars = plt.barh(y_pos, scores[::-1], color=np.array(bar_colors)[::-1])

# Y tick labels: feature names reversed so top is most important
plt.yticks(y_pos, np.array(feature_names)[::-1], fontsize=7 if n_feats > 80 else 9)
plt.gca().invert_yaxis()

plt.xlabel("importance_score")
plt.title("All features by importance_score (names on y-axis)")

# Add legend for categories (create proxy handles)
from matplotlib.patches import Patch
handles = []
seen = set()
for cat in cats:
    if cat not in seen:
        seen.add(cat)
        handles.append(Patch(color=color_map.get(cat, "#7f7f7f"), label=cat))
plt.legend(handles=handles, title="Category", bbox_to_anchor=(1.02, 1), loc='upper left')

plt.grid(axis='x', linestyle='--', alpha=0.3)

# Annotate top N bars with numeric score for clarity (only annotate the top-most ones)
if TOP_ANNOTATE and TOP_ANNOTATE > 0:
    n_annot = min(TOP_ANNOTATE, n_feats)
    # annotate the top features (they're at the beginning of imp_sorted)
    for i in range(n_annot):
        # bar index in plotted (reversed)
        bar_index = n_feats - 1 - i
        bar = bars[bar_index]
        score = scores[i]
        plt.text(
            bar.get_width() + 0.002,  # small offset
            bar.get_y() + bar.get_height()/2,
            f"{score:.3f}",
            va='center',
            fontsize=8
        )

# If too many labels, improve spacing and rotate ticks slightly if needed
plt.tight_layout()
plt.savefig(OUT_PLOT, dpi=150)
plt.show()

print(f"Saved plot with feature names to: {OUT_PLOT}")
print(f"Plotted {n_feats} features. Top 20 features:")
print(imp_sorted.head(20)[['importance_score', 'category_inferred']].to_string())


In [ ]:
"""
Prune the original features CSV by keeping only features whose importance_score >= threshold.
Saves a new CSV with the same index and target column(s) preserved so you can feed it to your NN.

Assumption: importance scores are in "feature_importance_summary.csv" and the original full features
file is "processed_features_conf_0.7.csv". By default we keep features with importance_score >= 0.10.
Change THRESHOLD to use a strict '>' instead of '>=' if you prefer strictly above 10%.
"""

import os
import re
import pandas as pd
import numpy as np

# ---------- CONFIG ----------
IMPORTANCE_CSV = "feature_importance_summary.csv"
FEATURES_CSV = "processed_features_conf_0.7.csv"
OUT_REDUCED = "processed_features_conf_0.7_reduced_importance_ge_80pct.csv"
THRESHOLD = 0.8      # keep features with importance_score >= THRESHOLD
# ----------------------------

if not os.path.exists(IMPORTANCE_CSV):
    raise FileNotFoundError(f"Importance CSV not found: {IMPORTANCE_CSV}")
if not os.path.exists(FEATURES_CSV):
    raise FileNotFoundError(f"Features CSV not found: {FEATURES_CSV}")

# Load importance table (handle either 'feature' column or index)
imp = pd.read_csv(IMPORTANCE_CSV)
if 'feature' in imp.columns:
    imp = imp.set_index('feature')
elif 'Unnamed: 0' in imp.columns and imp['Unnamed: 0'].dtype == object:
    imp = imp.set_index('Unnamed: 0')

if 'importance_score' not in imp.columns:
    raise ValueError("importance_score column missing from the importance CSV. Run the importance creation script first.")

# Decide which features to keep according to threshold
# Use >= by default; adjust comparison operator if you want strict >
keep_mask = imp['importance_score'] >= THRESHOLD
keep_features_from_imp = imp.loc[keep_mask].index.tolist()

print(f"Found {len(imp)} features in importance CSV, {len(keep_features_from_imp)} meet importance >= {THRESHOLD:.2%}")

# Load original features CSV (preserve index and all columns)
df = pd.read_csv(FEATURES_CSV, index_col=0)
orig_cols = df.columns.tolist()
print(f"Original features file shape: {df.shape} (columns: {len(orig_cols)})")

# Heuristic: detect target columns to always preserve them (do not drop)
possible_targets = [c for c in df.columns if re.search(r'safety|target|label|bin|class', c, flags=re.I)]
# prefer names containing 'safety' or 'bin' if multiple detected
if possible_targets:
    target_cols = []
    for prefer in ['safety', 'bin', 'label', 'target', 'class']:
        selected = [c for c in possible_targets if prefer in c.lower()]
        if selected:
            target_cols = selected
            break
    if not target_cols:
        target_cols = [possible_targets[0]]
else:
    target_cols = []

print("Detected target columns to preserve:", target_cols)

# Intersect features present in both importance CSV and original features CSV
common_feats = [f for f in keep_features_from_imp if f in orig_cols]
missing_in_orig = [f for f in keep_features_from_imp if f not in orig_cols]
if missing_in_orig:
    print(f"Warning: {len(missing_in_orig)} features selected by importance are not present in original CSV (they will be ignored).")
    print("Examples:", missing_in_orig[:10])

print(f"{len(common_feats)} features from importance list are present in the original file and will be kept.")

# Build the list of columns to keep: index is preserved automatically by read_csv(index_col=0)
cols_to_keep = list(target_cols)  # ensure targets come first (if any)
# Avoid duplicates
for f in common_feats:
    if f not in cols_to_keep:
        cols_to_keep.append(f)

# Defensive: if no features selected (rare), raise error
if len(cols_to_keep) == 0:
    raise RuntimeError("No columns selected to keep (no target columns and no features above threshold). Adjust THRESHOLD or check your files.")

# Make reduced dataframe and save
reduced_df = df[cols_to_keep].copy()
reduced_df.to_csv(OUT_REDUCED, index=True)
print(f"Saved reduced features CSV to: {OUT_REDUCED}")
print(f"Reduced CSV shape: {reduced_df.shape} (kept {len(cols_to_keep)} columns including {len(target_cols)} target cols)")

# Optional: print summary
dropped = [c for c in orig_cols if c not in cols_to_keep]
print(f"Dropped {len(dropped)} columns. Example dropped features (first 20): {dropped[:20]}")


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
# Import for class weight calculation and signal analysis
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report
from tensorflow.keras.regularizers import l2
import os

# --- Configuration ---
# NOTE: The script assumes FEATURE_FILE and TRUESKILL_FILE exist in the current directory
FEATURE_FILE = "processed_features_conf_0.7.csv"
TRUESKILL_FILE = "ai_rank_gpt.csv"
RANDOM_STATE = 42
TOTAL_CLASSES = 3

print(f"--- Starting Robust NN Training using {FEATURE_FILE} ---")

# 1. Load Data
try:
    X_df = pd.read_csv(FEATURE_FILE, index_col='image_name').fillna(0.0)
    ts_df = pd.read_csv(TRUESKILL_FILE).set_index('image_name')[['rank_ai']].rename(columns={'rank_ai': 'safety_score'})
except FileNotFoundError as e:
    print(f"\nFATAL ERROR: Required file not found. Ensure '{FEATURE_FILE}' and '{TRUESKILL_FILE}' exist and are loaded.")
    raise e

print(f"Feature DataFrame loaded (Shape: {X_df.shape}). NaN values successfully filled with 0.0.")

# 2. Target Variable (Safety Bin) Preparation
# Re-calculate the 10 safety bins to ensure consistency
minv = ts_df['safety_score'].min()
maxv = ts_df['safety_score'].max()
bins = np.linspace(minv, maxv, TOTAL_CLASSES + 1)   # 11 edges -> 10 equal-width bins

ts_df['safety_bin_1_10'] = pd.cut(
    ts_df['safety_score'],
    bins=bins,
    labels=range(1, TOTAL_CLASSES + 1),    # labels 1..10
    include_lowest=True
).astype(int)
counts = ts_df['safety_bin_1_10'].value_counts(sort=False).reindex(range(1,TOTAL_CLASSES + 1), fill_value=0)
print("Counts per equal-width safety bin (1..10):")
print(counts)

# 3. print bin ranges so you know what each label means
print("\nLabel -> numeric range (inclusive left, exclusive right except last):")
for i in range(TOTAL_CLASSES):
    left, right = bins[i], bins[i+1]
    # close the last interval on the right for readability
    right_bracket = ']' if i == 9 else ')'
    print(f"Label {i+1}: [{left:.6g}, {right:.6g}{right_bracket}")

# 4. bar plot (histogram of counts per label)
fig, ax = plt.subplots(figsize=(8,4))
counts.plot(kind='bar', ax=ax)
ax.set_xlabel('Safety bin (equal-width, 1 = lowest range)')
ax.set_ylabel('Number of features')
ax.set_title('Number of features in each equal-width safety bin (1..10)')
ax.set_xticklabels([str(i) for i in range(1,TOTAL_CLASSES + 1)], rotation=0)
plt.tight_layout()
plt.show()

# min_score = ts_df['safety_score'].min()
# max_score = ts_df['safety_score'].max()
# bins = np.linspace(min_score, max_score, TOTAL_CLASSES + 1)
# ts_df['safety_bin_1_10'] = pd.cut(
#     ts_df['safety_score'],
#     bins=bins,
#     labels=False,
#     include_lowest=True
# )
# The y_raw_full series contains the 0-9 labels indexed by image_name
y_raw_full = ts_df['safety_bin_1_10']

# 3. Final Merge and Alignment
combined_df = X_df.merge(y_raw_full, left_index=True, right_index=True, how='inner')

# Find the target column name
target_col_name = [col for col in combined_df.columns if 'safety_bin_1_10' in col][-1]

X = combined_df.drop(columns=[target_col_name]).values
y_raw = combined_df[target_col_name].values
y_raw = y_raw.astype(int) - 1
# 4. Feature Scaling
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print("\n[4/8] Features scaled using StandardScaler.")

# --- Data Split: 80% Train, 10% Validation, 10% Test ---

# 5. Split 1: 80% Train, 20% Temp (Test + Validation)
X_train, X_temp, y_train_raw, y_temp_raw = train_test_split(
    X_scaled, y_raw, test_size=0.2, random_state=RANDOM_STATE, stratify=y_raw
)

# 6. Split 2: Divide Temp (20%) into Test (10%) and Validation (10%) (0.5 means 50% of 20% = 10% total)
X_val, X_test, y_val_raw, y_test_raw = train_test_split(
    X_temp, y_temp_raw, test_size=0.5, random_state=RANDOM_STATE, stratify=y_temp_raw
)

# 7. One-Hot Encoding the Target Sets
y_train = to_categorical(y_train_raw, num_classes=TOTAL_CLASSES)
y_val = to_categorical(y_val_raw, num_classes=TOTAL_CLASSES)
y_test = to_categorical(y_test_raw, num_classes=TOTAL_CLASSES)
print("[7/8] Target variables One-Hot Encoded.")


# 8. Calculate Class Weights (Crucial for Imbalanced Data)
classes = np.unique(y_train_raw)
class_weights_array = compute_class_weight(
    'balanced',
    classes=classes,
    y=y_train_raw
)
class_weight_dict = dict(zip(classes, class_weights_array))
print("\n[8/8] Class weights calculated to handle data imbalance:")
print(class_weight_dict)

# --- Summary of Prepared Data ---
print("\n--- Summary of Prepared Data for Robust Training ---")
print(f"Training Features (X_train) Shape: {X_train.shape} (80%)")
print(f"Validation Features (X_val) Shape: {X_val.shape} (10%)")
print(f"Testing Features (X_test) Shape: {X_test.shape} (10%)")


# --- Robust Model Definition ---
def create_robust_model(input_dim, output_dim):
    model = Sequential([
        # Input Layer: Wider layer to capture initial feature interactions
        Dense(256, activation='relu', input_shape=(input_dim,)),
        Dropout(0.3),
        # Hidden Layer 1: Increased depth
        Dense(128, activation='relu'),
        Dropout(0.3),
        # Hidden Layer 2
        Dense(64, activation='relu'),
        # Output layer
        Dense(output_dim, activation='softmax')
    ])

    model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model


# Create and summarize the model
input_dim = X_train.shape[1]
output_dim = y_train.shape[1]
model = create_robust_model(input_dim, output_dim)

print("\n--- Robust Neural Network Model Summary ---")
model.summary()

# --- Callbacks for Robust Training ---
# Early Stopping: Stop training when validation loss stops improving (patience=10 epochs)
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=10,
    verbose=1,
    mode='min',
    restore_best_weights=True
)

# ReduceLROnPlateau: Reduce learning rate if val_loss plateaus
reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=5,
    min_lr=0.00001,
    verbose=1
)

callbacks = [early_stop, reduce_lr]

# --- Training the Robust Model ---
print("\n--- Starting Robust Model Training (Max 100 Epochs) ---")
history = model.fit(
    X_train, y_train,
    epochs=100, # Max epochs set high, relying on EarlyStopping
    batch_size=32,
    validation_data=(X_val, y_val), # Use the dedicated validation set
    # class_weight=class_weight_dict, # Apply calculated weights for imbalance
    callbacks=callbacks,
    verbose=1
)

# --- Final Evaluation on the Test Set ---
print("\n--- Final Evaluation on Dedicated 10% Test Set ---")
loss, accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f"Test Loss: {loss:.4f}")
print(f"Test Accuracy: {accuracy:.4f}")

# --- Comprehensive Classification Report ---
print("\n--- Comprehensive Classification Report (Test Set) ---")
y_pred_probs = model.predict(X_test)
y_test_labels = np.argmax(y_test, axis=1)
y_pred_labels = np.argmax(y_pred_probs, axis=1)

target_names = [f'Bin {i}' for i in range(1, TOTAL_CLASSES + 1)]
# Set zero_division=0 to handle cases where a bin has no predictions/true samples
print(classification_report(y_test_labels, y_pred_labels, target_names=target_names, labels=range(TOTAL_CLASSES), zero_division=0))

# --- PLOTTING THE CONFUSION MATRIX  ---
print("\n--- Plotting Confusion Matrix ---")

# Calculate the confusion matrix
cm = confusion_matrix(y_test_labels, y_pred_labels, labels=range(TOTAL_CLASSES))

# Plot the confusion matrix
fig, ax = plt.subplots(figsize=(TOTAL_CLASSES, TOTAL_CLASSES))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=target_names)
disp.plot(cmap=plt.cm.Blues, ax=ax)
plt.title('Confusion Matrix (Test Set)', fontsize=16)
plt.show()

# --- PLOTTING THE TRAINING CURVES  ---
print("\n--- Generating Training and Validation Curves ---")

# Extract metrics from the history object
train_acc = history.history['accuracy']
train_loss = history.history['loss']
val_acc = history.history['val_accuracy']
val_loss = history.history['val_loss']
epochs = range(1, len(train_acc) + 1)

# Find the best epoch based on minimum validation loss (for plotting marker)
best_epoch_index = np.argmin(val_loss)
best_epoch = best_epoch_index + 1

plt.figure(figsize=(14, 6))

# Subplot 1: Accuracy
plt.subplot(1, 2, 1)
plt.plot(epochs, train_acc, 'go-', label='Training Accuracy', linewidth=2, alpha=0.8)
plt.plot(epochs, val_acc, 'bo-', label='Validation Accuracy', linewidth=2, alpha=0.8)
plt.plot(best_epoch, val_acc[best_epoch_index], 'r*', markersize=15, label=f'Best Epoch ({best_epoch})')

plt.title('Training and Validation Accuracy', fontsize=16)
plt.xlabel('Epochs', fontsize=12)
plt.ylabel('Accuracy', fontsize=12)
plt.legend(loc='lower right')
plt.grid(True, linestyle='--', alpha=0.6)

# Subplot 2: Loss
plt.subplot(1, 2, 2)
plt.plot(epochs, train_loss, 'r^-', label='Training Loss', linewidth=2, alpha=0.8)
plt.plot(epochs, val_loss, 'c^-', label='Validation Loss', linewidth=2, alpha=0.8)
plt.plot(best_epoch, val_loss[best_epoch_index], 'm*', markersize=15, label=f'Best Epoch ({best_epoch})')

plt.title('Training and Validation Loss', fontsize=16)
plt.xlabel('Epochs', fontsize=12)
plt.ylabel('Loss (Categorical Crossentropy)', fontsize=12)
plt.legend(loc='upper right')
plt.grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()

print("\n--- Robust Training and Evaluation Complete ---")
print("Check the Test Accuracy and the Classification Report. The class weights should now help improve performance in the previously ignored bins.")

In [ ]:
import pandas as pd
import numpy as np
import os
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import load_model

from IPython.display import HTML

# --- Configuration ---
FEATURE_FILE = "processed_features_conf_0.7.csv"
TRUESKILL_FILE = "ai_rank_gpt.csv"
MODEL_FILE = "my_model.keras"
# Base URL for public image access
BASE_IMAGE_URL = "https://cycleai.net/london/training/training/Images/"
TOTAL_CLASSES = 3

print("--- Starting Inference Pipeline ---")

# 1. Load Data
try:
    X_df = pd.read_csv(FEATURE_FILE, index_col='image_name').fillna(0.0)
    ts_df = pd.read_csv(TRUESKILL_FILE).set_index('image_name')[['ai_random_ts_score']].rename(columns={'ai_random_ts_score': 'safety_score'})
except FileNotFoundError:
    print("Error: Input CSV files not found.")
    exit()

print(f"Diagnostic: Feature file (X_df) loaded with {len(X_df)} images.")
print(f"Diagnostic: TrueSkill file (ts_df) loaded with {len(ts_df)} scores.")

# 2. Replicate Training Processing (Crucial for consistency)
# We must perform the exact same binning to know what "Bin 0", "Bin 1", etc. represent.
try:
    minv = ts_df['safety_score'].min()
    maxv = ts_df['safety_score'].max()

    # 1. Re-calculate equal-width bin edges based on TOTAL_CLASSES
    # (TOTAL_CLASSES + 1) edges for TOTAL_CLASSES bins
    bin_edges = np.linspace(minv, maxv, TOTAL_CLASSES + 1)

    # 2. Use pd.cut to assign the labels to a NEW column named 'safety_bin'
    # We store the *labels* in the DataFrame, and the *edges* in bin_edges (defined above)
    ts_df['safety_bin'] = pd.cut(
        ts_df['safety_score'],
        bins=bin_edges,  # Use the calculated edges
        labels=range(1, TOTAL_CLASSES + 1), # labels 1, 2, 3
        include_lowest=True
    ).astype(int)

    # 3. Print counts for verification (optional, but good practice)
    counts = ts_df['safety_bin'].value_counts(sort=False).reindex(range(1, TOTAL_CLASSES + 1), fill_value=0)
    print(f"Recovered {len(bin_edges)-1} bins from training logic.")
    print(f"Bin edges: {np.round(bin_edges, 3)}")

except ValueError as e:
    print(f"Error reproducing binning logic: {e}")
    exit()

# 3. Fit Scaler on Training Data
# The scaler must be fit on the exact same data used to train the model.
# The inner merge finds the set of images that have both features AND valid labels.
combined_df = X_df.merge(ts_df[['safety_bin']], left_index=True, right_index=True, how='inner')

# Training features (X) used for fitting the scaler
X_train_raw = combined_df.drop(columns=['safety_bin']).values

scaler = StandardScaler()
scaler.fit(X_train_raw)
print(f"Scaler re-fitted on {len(X_train_raw)} labeled images (combined_df size).")

# 4. Select 10 Images for Prediction
# Unseen = Images in original Feature File (X_df) that were NOT part of the final training set (combined_df)
unseen_indices = X_df.index.difference(combined_df.index)

print(f"Diagnostic: Found {len(unseen_indices)} images that are in the feature file but not the training set.")

if len(unseen_indices) == 0:
    print("\nWARNING: No new/unseen images found. This means all images in your feature file also have a score.")
    print("Falling back to selecting 10 random images from the full feature set (X_df) to demonstrate the pipeline.")
    sample_indices = np.random.choice(X_df.index, size=10, replace=False)
elif len(unseen_indices) < 10:
    print(f"Not enough unseen images found (only {len(unseen_indices)}). Prediction will be run on these.")
    sample_indices = unseen_indices
else:
    # Randomly sample 10 unseen images
    sample_indices = np.random.choice(unseen_indices, size=10, replace=False)

print(f"\nSelected 10 images for prediction: {sample_indices}")

# 5. Prepare Features for the Sample
X_sample_raw = X_df.loc[sample_indices].values
X_sample_scaled = scaler.transform(X_sample_raw)

# 6. Load Model and Predict
try:
    if not os.path.exists(MODEL_FILE):
        print(f"Model file '{MODEL_FILE}' not found. Please ensure the training script ran successfully.")
        exit()

    model = load_model(MODEL_FILE)
    predictions_probs = model.predict(X_sample_scaled, verbose=0)

    # Get the predicted bin index
    predicted_bins = np.argmax(predictions_probs, axis=1)

    # Calculate Estimated Score based on qcut edges
    # We take the midpoint of the predicted bin's edges
    estimated_scores = []
    for bin_idx in predicted_bins:
        # Safety check: ensure predicted bin is within the edges we recovered
        if bin_idx < len(bin_edges) - 1:
            lower = bin_edges[bin_idx]
            upper = bin_edges[bin_idx + 1]
            midpoint = (lower + upper) / 2
            estimated_scores.append(midpoint)
        else:
            # Fallback for out-of-bounds prediction (e.g., if model predicts bin 9 but only 6 bins were created)
            print(f"Warning: Predicted bin {bin_idx} is outside the known range {len(bin_edges)-1}. Using max edge.")
            estimated_scores.append(bin_edges[-1])

except Exception as e:
    print(f"Error during inference: {e}")
    exit()

# 7. Display Results with Image URL in a formatted HTML table
print("\n--- Prediction Results (Visual Display) ---")

html_content = """
<style>
    .prediction-table { border-collapse: collapse; width: 100%; font-family: sans-serif; }
    .prediction-table th, .prediction-table td { border: 1px solid #ddd; padding: 8px; text-align: left; }
    .prediction-table th { background-color: #f2f2f2; color: #333; }
    .prediction-image { max-width: 150px; height: auto; border-radius: 4px; }
    .score-container { display: flex; align-items: center; }
</style>
<table class="prediction-table">
    <thead>
        <tr>
            <th>Image Name</th>
            <th>Predicted Bin</th>
            <th>Est. Safety Score</th>
            <th>Image Preview</th>
        </tr>
    </thead>
    <tbody>
"""

for i, img_name in enumerate(sample_indices):
    bin_idx = predicted_bins[i]
    score = estimated_scores[i]
    image_url = f"{BASE_IMAGE_URL}{img_name}"

    # Calculate color for visual representation (normalized 0 to 100, then mapped to a green/red gradient)
    # This is a heuristic mapping for visualization
    # We use a simple linear interpolation based on the score bounds recovered from qcut
    min_score = bin_edges[0]
    max_score = bin_edges[-1]

    # Normalize score between 0 and 1
    if max_score > min_score:
        normalized_score = (score - min_score) / (max_score - min_score)
    else:
        normalized_score = 0.5

    # Interpolate from Red (0, unsafe) to Green (1, safe)
    red = int(255 * (1 - normalized_score))
    green = int(255 * normalized_score)
    score_color = f"rgb({red}, {green}, 0)"

    html_content += f"""
        <tr>
            <td>{img_name}</td>
            <td>Bin {bin_idx}</td>
            <td>
                <div class="score-container">
                    <span style="background-color: {score_color}; color: white; padding: 3px 6px; border-radius: 4px; margin-right: 10px; font-weight: bold;">
                        {score:.4f}
                    </span>
                    <small>(Range: {bin_edges[bin_idx]:.2f} - {bin_edges[bin_idx+1]:.2f})</small>
                </div>
            </td>
            <td>
                <a href="{image_url}" target="_blank">
                    <img src="{image_url}" class="prediction-image" alt="Preview of {img_name}">
                </a>
            </td>
        </tr>
    """

html_content += """
    </tbody>
</table>
<p>(Note: 'Est. Safety Score' is the midpoint of the Quantile Bin from training. Click the image to view full size.)</p>
"""
display(HTML(html_content))